# AI Tutor — **Multi-turn Eval** · OSS Qwen + Gemma · Colab GPU

**Full tutoring sessions**: each scenario drives a complete 10–30-turn session —
an Anthropic-Haiku student simulator plays one of five personas (capable, average,
struggler, error-prone, non-responder) against the OSS tutor, and the whole
transcript is scored by deterministic assertions (turn budgets, repetition,
tool-syntax leaks, expected end-state) plus a session-level rubric.

- **Tutor** = OSS Qwen / Gemma via Ollama (swapped per model with `TUTOR_MODEL_OVERRIDE`).
- **Student sim** = Anthropic **Haiku 4.5** persona player (constant across models).
- **Judge** = Anthropic **Sonnet 4.6** @ temp 0 session-level rubric
  (`evals/runner.py` `MULTI_TURN_RUBRIC_JUDGE` — identical to the engine
  fix-cycle sweeps, so scores are comparable).
- **Engine** = `simple_tutor` (the production engine; `run_eval` prints a banner
  confirming it and errors on the legacy engine).
- **Benchmark draw** = the same 20-scenario seed-5 sample the engine fix-cycle
  board was measured on (gemini-2.5-flash 18/20 · kimi-k2-thinking 18/20 ·
  qwen3-next-80b 17/20 at cycle 11) — the OSS numbers here read directly
  against that board.

**Run one tab per GROUP (Colab Pro+ allows concurrent A100/H100 sessions).** The 13
OSS Qwen + Gemma models are split into 3 **disjoint size groups**; pick this tab's
group in the **Cell 7b** dropdown. Every tab writes per-model JSONs into the
**same** Drive folder, so no two tabs score the same model and the board merges
automatically:

- **group1 — small (≤4B)** — `gemma3:1b`, `gemma2:2b`, `qwen3:4b`, `qwen3.5:4b`, `gemma3:4b`
- **group2 — medium (9–14B)** — `gemma2:9b`, `qwen3.5:9b`, `gemma3:12b`, `qwen3:14b`
- **group3 — large (27B+)** — `gemma2:27b`, `gemma3:27b`, `qwen3:30b-a3b`, `qwen3.6:35b-a3b`

Each model auto-tunes to its family profile (Qwen → Markdown Block-0, temp 0.7 /
top_p 0.8 / top_k 20, num_ctx 24K so `<think>` + answer fit; Gemma runs the
profile registered in `apps/llm/model_profiles`, or engine defaults if none).

## Read this before budgeting a session

**Multi-turn is an order of magnitude heavier than single-turn.** Each scenario
is a full session: every turn costs 1–2 tutor calls (Ollama, GPU-bound) plus a
student-sim call and, at session end, one long-context Sonnet rubric call. On the
cloud sweeps a 20-scenario leg took ≈40 min for a fast API model and 2.5–4 h for
a slow thinking model; local Ollama inference speed will dominate here — expect
the small models to clear a leg in well under an hour on a T4/L4 and the 27B+
group to need a few hours on an A100.

**Scores read against the engine fix-cycle board.** Same 20 scenarios, same
seed, same student sim, same judge — the only variable is the tutor model. The
engine's guard stack (forced tool calls, pose salvage, letter coherence,
vocabulary scrub) is family-gated and active for Ollama models exactly as it was
for the cloud OSS models.

**Output → `offline_eval/multi_turn_results/oss13_mt/`** (symlinked to Drive, resume-safe).

> ⚠️ **Runtime.** Each model runs 20 full sessions of 10–30 turns; every
> turn is 1–2 Ollama tutor calls plus a Haiku student-sim call, and each session
> ends with one long-context Sonnet rubric call. Thinking models (which emit
> thousands of `<think>` tokens per turn) dominate wall-clock. The run is
> **resume-safe** (finished models are skipped, results live on Drive), so you
> can stop/restart freely.

**Before you start**
1. Runtime → **Change runtime type** → pick the GPU for THIS tab's group (High-RAM):
   **group1** (≤4B) T4/L4 ok · **group2** (9–14B) L4 or A100 · **group3** (27B+)
   **A100 40 GB** — the 27b dense and 35b-a3b MoE q4 weights (~17–22 GB) plus
   KV cache need more than a T4/L4 comfortably provides.
2. Add these **Colab Secrets** (🔑 sidebar), each *Notebook access ON*:
   - `GH_TOKEN` — GitHub **classic** PAT with **`repo`** scope (you're a collaborator on `eai6/ai-tutor`).
   - `ANTHROPIC_API_KEY` — **required** (student-sim Haiku 4.5 + Sonnet 4.6 session rubric judge).
   - `GOOGLE_API_KEY` + `OPENAI_API_KEY` — keep both so the cross-vendor **grader**
     cascade (Gemini→OpenAI→Haiku, self-excluding) matches the laptop runs.

## Cell 1 — confirm GPU + mount Drive (Drive persists results across disconnects)

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — clone the repo (branch `pixeldesignlabs-dev-portuguese`) using the GH_TOKEN classic PAT

In [ ]:
from google.colab import userdata
import subprocess, os
tok = (userdata.get('GH_TOKEN') or '').strip()   # strip stray spaces/newlines
assert tok and ' ' not in tok, "GH_TOKEN missing or contains a space — re-save the secret with no whitespace"
url = f"https://{tok}@github.com/eai6/ai-tutor.git"
subprocess.run(['rm', '-rf', '/content/ai-tutor'], check=True)
subprocess.run(['git', 'clone', '--depth', '1', '-b', 'pixeldesignlabs-dev-portuguese', url, '/content/ai-tutor'], check=True)
os.chdir('/content/ai-tutor')
print('cloned at', os.getcwd())

## Cell 3 — fix hardcoded laptop paths (essential)

In [ ]:
!sed -i 's#/home/daniel/Documents/work/Nyansapo/web/ai-tutor#/content/ai-tutor#g; s#\$ROOT/venv/bin/python#python#g; s#venv/bin/python#python#g' offline_eval/*.py offline_eval/*.sh

## Cell 4 — install deps + start Ollama (a few min; ignore pip resolver warnings)

In [ ]:
!pip install -q -r requirements.txt
# The Ollama installer is now zstd-compressed; the Colab VM lacks zstd, so install
# it first (otherwise the installer aborts and `ollama` is never created).
!apt-get -qq install -y zstd || (apt-get -qq update && apt-get -qq install -y zstd)
import subprocess, time, shutil, os
# Robust install: the official installer pulls the binary from github releases,
# which intermittently times out on Colab VMs. Retry it, then fall back to the
# standalone linux binary straight from ollama.com (no github).
def _has_ollama():
    return shutil.which('ollama') is not None
def _install_ollama():
    if _has_ollama():
        return True
    for i in range(1, 4):
        print(f'[ollama] install.sh attempt {i}', flush=True)
        subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True)
        if _has_ollama():
            return True
        time.sleep(5)
    for i in range(1, 4):
        print(f'[ollama] direct-binary attempt {i} (ollama.com)', flush=True)
        subprocess.run('curl -fL --retry 5 --retry-all-errors --connect-timeout 30 '
                       '-o /tmp/ollama.tgz https://ollama.com/download/ollama-linux-amd64.tgz',
                       shell=True)
        if os.path.exists('/tmp/ollama.tgz') and os.path.getsize('/tmp/ollama.tgz') > 1_000_000:
            subprocess.run('tar -C /usr -xzf /tmp/ollama.tgz', shell=True)
            if _has_ollama():
                return True
        time.sleep(5)
    return False
assert _install_ollama(), ("ollama install failed — Colab couldn't reach ollama.com/github. "
                           "Try Runtime -> Disconnect and delete runtime, then start fresh.")
subprocess.Popen(['ollama', 'serve'],
                 stdout=open('/content/ollama.log', 'w'),
                 stderr=subprocess.STDOUT)
for _ in range(30):
    if subprocess.run(['bash', '-c', 'ollama list'], capture_output=True).returncode == 0:
        print('ollama ready'); break
    time.sleep(2)
else:
    print('ollama NOT ready — check /content/ollama.log')

## Cell 5 — **required** — write .env from Colab Secrets
`.env` isn't in the repo (gitignored). Keep **all three** keys: ANTHROPIC drives the rubric judge (Haiku 4.5); GOOGLE/OPENAI feed the cross-vendor grader cascade so scores match the laptop runs.

In [ ]:
from google.colab import userdata
open('.env', 'w').write(
    "SECRET_KEY=colab-eval\nDEBUG=True\nEMBEDDING_BACKEND=sqlite\n"
    f"ANTHROPIC_API_KEY={userdata.get('ANTHROPIC_API_KEY')}\n"
    f"GOOGLE_API_KEY={userdata.get('GOOGLE_API_KEY')}\n"
    f"OPENAI_API_KEY={userdata.get('OPENAI_API_KEY')}\n")
print('.env written')

## Cell 6 — fresh DB + eval fixtures
`lessons.json` carries the 4 eval lessons with their **LessonSteps + exit tickets** (16 lessons: 8 math x 10 steps, 8 geography x 5) — the scenarios ground step, so this fixture is required. `institution.json` seeds the active `student_sim` ModelConfig (Haiku).

In [ ]:
!python manage.py migrate
!python manage.py loaddata evals/fixtures/institution.json evals/fixtures/lessons.json

## Cell 7 — persist results to Drive (symlink → survives disconnects)
Symlinks **only `offline_eval/multi_turn_results/oss13_mt/`** to `ai-tutor-eval-multiturn/oss13_mt/` on Drive. Resume-safe: a reconnect re-symlinks the same Drive folder and `run_matrix.sh` skips models that already have a JSON in `oss13_mt/`.

In [ ]:
!mkdir -p /content/drive/MyDrive/ai-tutor-eval-multiturn/oss13_mt
!rm -rf offline_eval/multi_turn_results/oss13_mt && ln -s /content/drive/MyDrive/ai-tutor-eval-multiturn/oss13_mt offline_eval/multi_turn_results/oss13_mt
import os, glob
print('this sweep writes to:', os.path.realpath('offline_eval/multi_turn_results/oss13_mt'))
done = sorted(os.path.basename(p)[:-5] for p in glob.glob('offline_eval/multi_turn_results/oss13_mt/*.json'))
print('already scored in oss13_mt:', done or '(none yet)')

## Cell 7b — **pick this tab's model GROUP** (run one tab per group)
Set `GROUP` in the dropdown (right side in Colab) to `group1`, `group2`, or `group3`. The groups are **disjoint, size-based** sets; every tab writes to the same Drive folder so the board merges. Re-run this cell whenever you change it.

- **group1 — small (≤4B)** — `gemma3:1b`, `gemma2:2b`, `qwen3:4b`, `qwen3.5:4b`, `gemma3:4b`
- **group2 — medium (9–14B)** — `gemma2:9b`, `qwen3.5:9b`, `gemma3:12b`, `qwen3:14b`
- **group3 — large (27B+)** — `gemma2:27b`, `gemma3:27b`, `qwen3:30b-a3b`, `qwen3.6:35b-a3b`

In [ ]:
GROUP = "group2"  #@param ["group1", "group2", "group3"]
print("This tab will evaluate group:", GROUP)

## Cell 8 — write THIS tab's matrix (from GROUP) + seed configs
Writes only the models for the group picked in **Cell 7b** into `models.txt`, then prints the per-family sampling each will use. `run_matrix.sh` (Cell 9) reads this file. Resume-safe: any model already scored on Drive is skipped.

In [ ]:
# Master registry (tag -> tier, note) + the 3 disjoint size groups. Cell 7b's
# GROUP picks which models THIS tab writes into models.txt. Groups partition
# the 13 OSS Qwen + Gemma models so parallel Colab tabs never score the same
# model. group1 = small (<=4B), group2 = medium (9-14B), group3 = large (27B+).
MODELS = {
    # ── small (≤4B) ──
    'gemma3:1b':       ('big', ''),
    'gemma2:2b':       ('big', ''),
    'qwen3:4b':        ('big', ''),
    'qwen3.5:4b':      ('big', ''),
    'gemma3:4b':       ('big', ''),
    # ── medium (9–14B) ──
    'gemma2:9b':       ('big', ''),
    'qwen3.5:9b':      ('big', ''),
    'gemma3:12b':      ('big', ''),
    'qwen3:14b':       ('big', ''),
    # ── large (27B+; MoE tags sized by total params) ──
    'gemma2:27b':      ('xl',  'Gemma 2 dense 27B'),
    'gemma3:27b':      ('xl',  'Gemma 3 dense 27B'),
    'qwen3:30b-a3b':   ('xl',  'Qwen3 30B MoE (3B active) — fast for its size'),
    'qwen3.6:35b-a3b': ('xl',  'Qwen3.6 35B MoE (3B active)'),
}
GROUPS = {
    'group1': ['gemma3:1b', 'gemma2:2b', 'qwen3:4b', 'qwen3.5:4b',
               'gemma3:4b'],
    'group2': ['gemma2:9b', 'qwen3.5:9b', 'gemma3:12b', 'qwen3:14b'],
    'group3': ['gemma2:27b', 'gemma3:27b', 'qwen3:30b-a3b',
               'qwen3.6:35b-a3b'],
}
GROUP = globals().get('GROUP')
if GROUP is None:
    raise SystemExit("Run Cell 7b first to pick GROUP (group1/group2/group3).")
assert GROUP in GROUPS, f"GROUP must be one of {list(GROUPS)}, got {GROUP!r}"

lines = [f"# Single-turn Eval (sweep 3) — {GROUP} (this tab)"]
for tag in GROUPS[GROUP]:
    tier, note = MODELS[tag]
    lines.append(f"{tag:<20} {tier}" + (f"     # {note}" if note else ""))
open('offline_eval/models.txt', 'w').write("\n".join(lines) + "\n")
print(f">> {GROUP}: this tab will evaluate")
for tag in GROUPS[GROUP]:
    print("   -", tag)
print()

!python offline_eval/seed_ollama_configs.py
# Show the per-family sampling each model will use (from apps/llm/model_profiles).
import django, os
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings')
django.setup()
from apps.llm.model_profiles import get_model_profile
print(f"{'MODEL':<22} {'FAMILY':<9} {'MODE':<11} {'MAXTOK':>7}  SAMPLING")
print('-' * 78)
for line in open('offline_eval/models.txt'):
    tag = line.split('#')[0].split()[0] if line.split('#')[0].split() else ''
    if not tag:
        continue
    p = get_model_profile(f'local_ollama/{tag}')
    if p:
        print(f"{tag:<22} {p.family:<9} {p.mode:<11} {p.max_tokens:>7}  {p.sampling_dict()}")
    else:
        print(f"{tag:<22} (no profile — runs at engine default)")

## Cell 8b — reclaim disk BEFORE the sweep (important on a resumed session)
A reconnected / re-cloned Colab can start with old pulled model weights + caches still on disk. This clears every previously-pulled Ollama model and the package caches **before** the first download, then prints free space. On a fresh VM it's a harmless no-op.

In [ ]:
import subprocess
def _df(tag):
    print(f"--- disk {tag} ---\n" + subprocess.run(['df','-h','/'],capture_output=True,text=True).stdout)
_df('BEFORE cleanup')
# remove every model the Ollama server currently holds (server-mediated → frees blobs)
!ollama list 2>/dev/null | tail -n +2 | awk '{print $1}' | xargs -r -n1 ollama rm 2>/dev/null || true
# backstop: wipe any stray model store + pip/apt/HF caches (server re-creates on next pull)
!rm -rf /root/.ollama/models/blobs/* /root/.ollama/models/manifests/* offline_eval/ollama_models/* 2>/dev/null || true
!pip cache purge 2>/dev/null || true
!apt-get clean 2>/dev/null || true
!rm -rf /root/.cache/huggingface /root/.cache/pip 2>/dev/null || true
_df('AFTER cleanup')

## Cell 9 — run the MULTI-TURN sweep (pulls + scores each model; resume-safe)
`MODE="--multi-turn --sample 20 --seed 5"` runs 20 full sessions per model. `RESULTS_DIR=…/oss13_mt` writes to Drive. `CLEANUP_MODELS=1` deletes each model's weights right after it's scored so peak disk ≈ one model at a time. The run tolerates disconnects (done models are skipped on restart).

**Sanity-check the first model's output before walking away:**
- `>> Tutor engine: simple_tutor` — not the legacy engine.
- The runner reports 20 scenarios; if it reports 200, `SAMPLE` is None and you are doing the full publication run (~10x wall-clock).
- Sessions end with `exit_ticket` / `completed` — a wall of `deadlock` on the first model means something is wrong; stop and check the log.
- No `dropped duplicate pose_question` storms, and few `record_answer without in-flight question` lines.

In [ ]:
!RESULTS_DIR=$PWD/offline_eval/multi_turn_results/oss13_mt SIMPLE_TUTOR_ENGINE=1 CLEANUP_MODELS=1 \
  MODE="--multi-turn --sample 20 --seed 5" bash offline_eval/run_matrix.sh

## Cell 9b — reclaim disk AFTER the sweep (final backstop)
`CLEANUP_MODELS=1` already removes each model as it finishes; this drops anything left and prints free space. Results are safe on Drive (Cell 7).

In [ ]:
!ollama list 2>/dev/null | tail -n +2 | awk '{print $1}' | xargs -r -n1 ollama rm 2>/dev/null || true
!rm -rf /root/.ollama/models/blobs/* /root/.ollama/models/manifests/* offline_eval/ollama_models/* 2>/dev/null || true
!pip cache purge 2>/dev/null || true
!apt-get clean 2>/dev/null || true
import subprocess
print(subprocess.run(['df','-h','/'],capture_output=True,text=True).stdout)

## Cell 10 — multi-turn leaderboard (run anytime; scores whatever is in `oss13_mt/`)
Pass rates are out of **20** sessions. Every model ran the same seeded draw as the engine fix-cycle board, so these numbers read directly against gemini-2.5-flash 18/20, kimi-k2-thinking 18/20, and qwen3-next-80b 17/20 (cycle 11). An `errored` scenario is judge/sim infrastructure, not model failure — re-run those before comparing.

In [ ]:
!RESULTS_DIR=$PWD/offline_eval/multi_turn_results/oss13_mt python offline_eval/aggregate.py

## Cell 10b — error bars
A pass rate without an interval invites over-reading. At n=20 the standard error at p≈0.5 is ±11.2pp, so two models must differ by roughly 31pp before the gap means anything. This cell prints a Wilson 95% interval per model — read the intervals, not the point estimates.

In [ ]:
import json, glob, os, math

def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k / n
    d = 1 + z*z/n
    c = p + z*z/(2*n)
    s = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (100*(c - s)/d, 100*(c + s)/d)

paths = [p for p in sorted(glob.glob('offline_eval/multi_turn_results/oss13_mt/*.json'))
         if not os.path.basename(p).startswith('_')]
rows = []
for p in paths:
    m = os.path.basename(p)[:-5]
    R = json.load(open(p))['results']
    k, n = sum(bool(r['passed']) for r in R), len(R)
    lo, hi = wilson(k, n)
    rows.append((m, k, n, 100*k/n, lo, hi))

print(f'{"MODEL":<28} {"PASS":>9}  {"RATE":>6}   95% CI')
print('-'*66)
for m, k, n, rate, lo, hi in sorted(rows, key=lambda r: -r[3]):
    print(f'{m:<28} {k:>4}/{n:<4} {rate:>5.0f}%   [{lo:>4.0f}, {hi:>4.0f}]')

## Cell 10c — session end-reasons (the multi-turn failure signature)
Pass rate alone hides HOW sessions end. `exit_ticket`/`completed` are clean; `max_turns` = pacing; `deadlock` = the tutor repeated itself verbatim — the classic weak-model failure the guard stack exists to prevent. `errored` is infrastructure (judge/sim overload): re-run those scenarios before comparing.

In [ ]:
import json, glob, os
from collections import Counter
print(f"{'MODEL':<26}{'PASS':>8}   SESSION END-REASONS")
print('-' * 70)
for f in sorted(glob.glob('offline_eval/multi_turn_results/oss13_mt/*.json')):
    name = os.path.basename(f)[:-5]
    if name.startswith('_'):
        continue
    d = json.load(open(f))
    ends = Counter(r.get('sim_reason') or ('errored' if r.get('error') else '?')
                   for r in d['results'])
    reasons = '  '.join(f"{k}:{v}" for k, v in ends.most_common())
    print(f"{name:<26}{d['passed']:>4}/{d['total_scenarios']:<3}  {reasons}")

## After a Colab disconnect (A100 Pro: long sessions, but not infinite)
Re-run **Cells 1–8b**, then **Cell 9** again. Because results live on Drive (Cell 7),
`run_matrix.sh` **skips already-scored models** and continues. To pick up new commits
on the branch, re-run **Cell 2** (it re-clones).

**Strategy for the slow tail.** The small + MoE models finish first (Cell 8 order).
If the 27b dense models are too slow for your session budget, just stop —
you'll still have a complete small/MoE board, and you can resume the big ones in a
later session (or trim group3 by commenting models out in Cell 8).

To pull results back to your laptop: copy the JSONs + logs from
`MyDrive/ai-tutor-eval-multiturn/oss13_mt/` into the repo's `offline_eval/multi_turn_results/oss13_mt/` and run
`RESULTS_DIR=offline_eval/multi_turn_results/oss13_mt python offline_eval/aggregate.py`.

**Sanity checks per model:**
- The `run_eval` banner reads `>> Tutor engine: simple_tutor` (NOT conversational_tutor).
- It reports `=== Running 20 scenario(s) ===`. If the count is wrong, the clone
  is stale — re-run **Cell 2**.
- `oss13_mt/<model>.log` shows sessions reaching `exit_ticket` / `completed`
  (not all `deadlock`); each scored session's `rubric_result.model` is
  `claude-sonnet-4-6`.
- Reasoning models: `[OllamaTools] response: ... blocks=['tool_use', ...]` (or `['text']`),
  **not** `blocks=[]` (empty = num_ctx still truncating).

**Log greps worth running per model (the guard stack should show up):**
- `dropped duplicate pose_question` — the cap firing. A few is fine; a storm means a
  model is spamming parallel calls (that was gemini-3.1-pro's 139-per-turn bug).
- `record_answer without in-flight question` — should be **rare**. In sweep 1 this hit
  90% of qwen3.5:4b's grading attempts.
- `call2_repair: Call 1 skipped …` — the repair riding on the second call. Expected on
  Ollama models, which cannot honour forced tool choice.
- `grading a STALE slot` — new diagnostic. Grep it afterwards; it tells us whether
  stale-question grading is a real problem or a rare one.